# 45. Delta-method error propagation

**Objectives:**
- Fit a small toy model and take Minuit's postfit covariance (`result.covariance`).
- Propagate that covariance through a custom JAX-differentiable derived observable (the ratio of
  two fitted coefficients) using `delta_method_jacobian`/`delta_method_covariance`/
  `delta_method_errors`.
- Cross-check against `FitSession.fit_fraction_errors`, which uses the same delta-method
  machinery internally on a real physics observable (fit fractions).

Run cells in order in a fresh kernel. See `docs/fitting.md` ("Fit fractions (delta method)").


In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant, Parameter, RealImag,
    Resonance, delta_method_covariance, delta_method_errors, delta_method_jacobian,
    generate_toy,
)

## 1. Fit a small toy model

The rho(770) + non-resonant model from earlier lessons, with `NR.x`/`NR.y` floating.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=60, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

data = generate_toy(model, 2500, parameters=truth, seed=2026, inverse_resolution=256)

session = FitSession(model, data)
start = {"NR.x": 0.35, "NR.y": 0.45}
result = session.fit(start, simplex=True, ncall=5000)
assert result.valid, "Inspect the fit diagnostics before using this result."
fit_values = session.result_values(result)
print({name: round(fit_values[name], 4) for name in ("NR.x", "NR.y")})

{'NR.x': 0.5666, 'NR.y': 0.3094}


## 2. A custom derived observable: `NR.x / NR.y`

`delta_method_jacobian` differentiates any pure-JAX function of the full parameter mapping with
reverse-mode autodiff. Here the derived quantity is a plain ratio of two floating coefficients --
deliberately simple, to see the propagated error against a hand-computable check.

In [3]:
def ratio(values):
    return values["NR.x"] / values["NR.y"]

free_names = [p.name for p in model.parameters if not p.fixed]
print("floating parameters:", free_names)

jacobian = delta_method_jacobian(ratio, fit_values, free_names)
covariance = delta_method_covariance(ratio, fit_values, free_names, result.covariance)
errors = delta_method_errors(ratio, fit_values, free_names, result.covariance)

print("Jacobian d(ratio)/d(parameters) =", np.asarray(jacobian))
print(f"ratio = {ratio(fit_values):.4f} +/- {float(errors[0]):.4f}")

floating parameters: ['NR.x', 'NR.y']


Jacobian d(ratio)/d(parameters) = [[ 3.23211542 -5.91856073]]
ratio = 1.8312 +/- 0.1253


## 3. Cross-check the Jacobian by hand

For `r = a/b`, `dr/da = 1/b` and `dr/db = -a/b^2`. Since `ratio` only depends on `NR.x` and
`NR.y`, every other column of the Jacobian (with respect to the other floating parameters, if
any) must be exactly zero.

In [4]:
a, b = fit_values["NR.x"], fit_values["NR.y"]
expected = {"NR.x": 1.0 / b, "NR.y": -a / b**2}

jac_row = np.asarray(jacobian)[0]
for name, expected_value in expected.items():
    index = free_names.index(name)
    print(f"d(ratio)/d({name}): autodiff={jac_row[index]:.6f}  hand={expected_value:.6f}")
    assert abs(jac_row[index] - expected_value) < 1e-8

for index, name in enumerate(free_names):
    if name not in expected:
        assert abs(jac_row[index]) < 1e-12

d(ratio)/d(NR.x): autodiff=3.232115  hand=3.232115
d(ratio)/d(NR.y): autodiff=-5.918561  hand=-5.918561


## 4. Cross-check against `FitSession.fit_fraction_errors`

`fit_fraction_errors` differentiates `PreparedAmplitudeCache.fit_fractions` with exactly the same
`delta_method_jacobian`/`delta_method_covariance` machinery used above, just on a different
(physical) output vector. Recompute one component's fit-fraction error directly with
`delta_method_errors` and confirm it matches the session-level convenience method.

In [5]:
fractions = session.print_fit_fractions(result)
errors_via_session = session.fit_fraction_errors(result)
print("fit fractions:", {k: round(v, 4) for k, v in fractions.items()})
print("fit fraction errors (session):", {k: round(v, 5) for k, v in errors_via_session.items()})

cache = model._fraction_cache(None, None)
component_names = [component.name for component in cache.components]

def fit_fraction_vector(values):
    return cache.fit_fractions(values)

direct_errors = delta_method_errors(fit_fraction_vector, fit_values, free_names, result.covariance)
direct_errors = {name: float(direct_errors[i]) for i, name in enumerate(component_names)}
print("fit fraction errors (direct delta_method_errors):", {k: round(v, 5) for k, v in direct_errors.items()})

for name in component_names:
    assert abs(direct_errors[name] - errors_via_session[name]) < 1e-8

Fit fractions (physical)
component                    fraction [%]
rho                                70.586
NR                                 29.414
sum                               100.000


fit fractions: {'rho': 0.7059, 'NR': 0.2941}
fit fraction errors (session): {'rho': 0.01424, 'NR': 0.01424}


fit fraction errors (direct delta_method_errors): {'rho': 0.01424, 'NR': 0.01424}


## Summary and exercises

1. Try a nonlinear derived observable, e.g. `NR.x**2 + NR.y**2` (the squared modulus of the NR
   coefficient), and note the delta-method error is still a *linear* approximation around the
   postfit point -- accurate when the postfit uncertainty is small compared to the curvature
   scale of the function, understated otherwise (see the `delta_method_errors` docstring).
2. Pass a dense NumPy covariance matrix (already ordered like `free_names`) instead of
   `result.covariance` directly to `delta_method_covariance` and confirm the same answer --
   `_covariance_matrix` accepts either an `iminuit`-style name-indexable object or a plain dense
   array in that exact order.
3. Compare the fit-fraction delta-method errors above against a small ensemble of independent
   toy fits (refit `session` on several toys and take the spread of `fit_fractions`) to see how
   well the linear approximation tracks the true sampling distribution at this event count.

Reference: `docs/fitting.md` ("Fit fractions (delta method)").

Return to [the course guide](TUTORIALS.md).
